# Phase 7: Occlusion Detection & Inpainting

**Pipeline position:**
```
Phase 1–6: Data through Texture  ✓
Phase 7: Occlusion Inpainting    ← YOU ARE HERE
Phase 8: Evaluation
```

---

## What Is Occlusion in Aerial Photogrammetry?

After Phase 6, some regions of the texture atlas are **unfilled** — no camera had a valid view of those parts of the mesh surface.  This is structural occlusion inherent to aerial acquisition:

| Cause | Example |
|---|---|
| **Building north faces** | Camera grid is south of the building; the north wall is never directly visible |
| **Building-behind-building** | A tall building blocks the base of a shorter one behind it |
| **Steep slopes** | Camera angle is too oblique; score < threshold for all cameras |
| **UV packing gaps** | Small numerical gaps between UV islands in the atlas |

Unfilled regions appear **black** in the atlas and produce ugly dark patches in the final 3-D model.  We need to fill them.

---

## Why Inpainting Works

Inpainting fills a hole in an image using information from its boundary.  It works because:
- The boundary of the hole contains real texture (surrounding visible faces)
- Building surfaces have smooth, slowly varying texture (walls, concrete, brick)
- The human eye is very tolerant of plausible infill in regions it never sees directly (the back of a building)

There are two approaches here:

### Classical: OpenCV Telea Algorithm

Uses the **Fast Marching Method (FMM)**: propagate inward from the hole boundary, filling each pixel as a weighted sum of known neighbours:

$$I(\mathbf{p}) = \frac{\displaystyle\sum_{\mathbf{q} \in B_\epsilon(\mathbf{p})} w(\mathbf{p}, \mathbf{q})\, \bigl[I(\mathbf{q}) + \nabla I(\mathbf{q}) \cdot (\mathbf{p} - \mathbf{q})\bigr]}{\displaystyle\sum_{\mathbf{q}} w(\mathbf{p}, \mathbf{q})}$$

The weight $w$ is higher for pixels closer to $\mathbf{p}$ and more aligned with the boundary normal.  This propagates colour **and gradient** — producing smooth transitions without blocky borders.

**Best for:** small, thin gaps (UV island borders, narrow occlusion lines).

### Neural: LaMa (Large Mask Inpainting)

A deep network trained specifically for large-hole inpainting.  It uses Fourier convolutions to capture global structure and can plausibly complete large missing regions (entire walls, rooftop spans).

**Best for:** large holes (north faces of buildings).  Requires `pip install simple-lama-inpainting`.

---

## Detection Strategy

We detect occluded regions as atlas pixels where the sum of RGB values is below a threshold (effectively: unpainted = black = 0):

$$\text{mask}(u,v) = \mathbf{1}\bigl[I_R(u,v) + I_G(u,v) + I_B(u,v) < \tau\bigr]$$

A small threshold (e.g. $\tau = 30$) avoids false-positives from legitimately very dark surfaces (black asphalt, shadowed areas).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import cv2
import open3d as o3d
import matplotlib.pyplot as plt
import json
%matplotlib inline

In [ ]:
atlas_path = os.path.join('..', 'outputs', 'texture_atlas.png')
mesh_path  = os.path.join('..', 'outputs', 'mesh.ply')
poses_path = os.path.join('..', 'data', 'synthetic', 'GT_poses.json')
assert os.path.exists(atlas_path), "Run phase6_texture.ipynb first."

atlas = cv2.imread(atlas_path)
with open(poses_path) as f:
    poses = json.load(f)
mesh = o3d.io.read_triangle_mesh(mesh_path) if os.path.exists(mesh_path) else None

coverage_pct = (atlas.sum(axis=2) > 0).mean() * 100
print(f"Atlas shape    : {atlas.shape}")
print(f"Filled pixels  : {coverage_pct:.1f}%")
print(f"Unfilled pixels: {100 - coverage_pct:.1f}%  (occlusion + UV packing gaps)")

---

## Step 1 — Occlusion Detection

We build the binary occlusion mask.  The visualisation overlays detected occluded regions in red on the atlas — these are the pixels we will inpaint.

In [ ]:
from src.occlusion.detector import detect_occlusion_mask

mask = detect_occlusion_mask(atlas, mesh, poses)

occ_pct = (mask > 0).mean() * 100
print(f"Occluded atlas area: {occ_pct:.1f}%")

# Analyse hole size distribution
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
hole_areas = stats[1:, cv2.CC_STAT_AREA]   # skip background label 0
print(f"Number of distinct holes: {len(hole_areas)}")
if len(hole_areas):
    print(f"Hole area distribution:")
    print(f"  min  = {hole_areas.min()} px²")
    print(f"  max  = {hole_areas.max()} px²")
    print(f"  mean = {hole_areas.mean():.0f} px²")
    print(f"  Holes > 1000 px²: {(hole_areas > 1000).sum()}  (large → LaMa recommended)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(atlas, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Atlas\n(black regions = unfilled)')
axes[0].axis('off')

axes[1].imshow(mask, cmap='Reds')
axes[1].set_title(f'Occlusion Mask\n({occ_pct:.1f}% of atlas is occluded)')
axes[1].axis('off')

# Overlay
overlay = cv2.cvtColor(atlas.copy(), cv2.COLOR_BGR2RGB)
overlay[mask > 0] = [220, 50, 50]
axes[2].imshow(overlay)
axes[2].set_title('Occluded Regions (red overlay)\non the atlas')
axes[2].axis('off')

plt.suptitle('Occlusion Detection Result', fontsize=12)
plt.tight_layout()
plt.show()

if len(hole_areas):
    plt.figure(figsize=(7, 3))
    plt.hist(hole_areas, bins=40, color='steelblue', edgecolor='none', log=True)
    plt.axvline(1000, color='red', linestyle='--', label='1000 px² (large-hole boundary)')
    plt.xlabel('Hole area (px²)')
    plt.ylabel('Count (log scale)')
    plt.title('Distribution of Occlusion Hole Sizes')
    plt.legend()
    plt.tight_layout()
    plt.show()

---

## Step 2 — Inpainting

We apply OpenCV Telea inpainting.  Key parameter: `inpaintRadius=5` controls how far from the hole boundary to look for colour samples.  Too small = blocky fill.  Too large = blurring across unrelated regions.

For large holes (> ~10,000 px²), Telea will produce blurry, washed-out fills.  That is acceptable for regions that are never directly viewed (building backs), but LaMa produces significantly better results when visual quality matters.

In [ ]:
from src.occlusion.inpainter import inpaint_atlas

print("Inpainting with OpenCV Telea...")
inpainted = inpaint_atlas(atlas, mask, use_lama=False)

out_path = os.path.join('..', 'outputs', 'texture_atlas_inpainted.png')
cv2.imwrite(out_path, inpainted)

remaining_black = (inpainted.sum(axis=2) == 0).mean() * 100
print(f"Saved: {out_path}")
print(f"Remaining unfilled after inpainting: {remaining_black:.2f}%  (should be ~0)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(atlas, cv2.COLOR_BGR2RGB))
axes[0].set_title('Before: Original Atlas')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(inpainted, cv2.COLOR_BGR2RGB))
axes[1].set_title('After: Telea Inpainting')
axes[1].axis('off')

# Difference image (amplified)
diff = np.abs(inpainted.astype(int) - atlas.astype(int)).astype(np.uint8)
diff_amp = np.clip(diff * 5, 0, 255).astype(np.uint8)
axes[2].imshow(cv2.cvtColor(diff_amp, cv2.COLOR_BGR2RGB))
axes[2].set_title('Changed pixels (5× amplified)\nOnly the masked holes changed')
axes[2].axis('off')

plt.suptitle('Occlusion Inpainting: Before vs. After', fontsize=12)
plt.tight_layout()
plt.show()

changed_pct = (diff.sum(axis=2) > 0).mean() * 100
print(f"Changed pixels: {changed_pct:.2f}%  (should match the mask area {occ_pct:.2f}%)")

### Visual Quality Assessment

Zoom into a representative inpainted region to see the quality of the fill:

In [ ]:
# Find the largest hole and zoom in
if len(hole_areas) > 0:
    largest_label = 1 + np.argmax(hole_areas)
    ys, xs = np.where(labels == largest_label)
    # Bounding box with margin
    margin = 40
    y0, y1 = max(0, ys.min()-margin), min(atlas.shape[0], ys.max()+margin)
    x0, x1 = max(0, xs.min()-margin), min(atlas.shape[1], xs.max()+margin)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(cv2.cvtColor(atlas[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Largest hole — before\n({hole_areas.max()} px², {hole_areas.max()/2048**2*100:.1f}% of atlas)')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(inpainted[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))
    axes[1].set_title('Largest hole — after Telea inpainting')
    axes[1].axis('off')

    plt.suptitle('Zoom on Largest Occluded Region', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No holes detected — atlas is fully covered.")

---

## Summary

| Step | Method | Notes |
|---|---|---|
| Occlusion detection | Dark-pixel threshold ($\sum RGB < \tau$) | Simple, works for photogrammetry atlases |
| Hole analysis | Connected components | Informs algorithm choice |
| Inpainting (default) | OpenCV Telea (FMM) | Fast, good for small holes |
| Inpainting (optional) | LaMa (neural) | Better for large holes, needs GPU |

### LaMa Installation (Optional)

```bash
pip install simple-lama-inpainting
```
Then re-run with `use_lama=True`:
```python
inpainted = inpaint_atlas(atlas, mask, use_lama=True)
```

### Questions to Think About

- The detection threshold $\tau = 30$ avoids flagging dark-but-valid pixels as holes.  What would happen if a building in the scene was painted jet-black?
- Telea propagates colour from the boundary inward.  For a circular hole 200 px in diameter, how many "layers" of propagation are needed to fill the centre?
- Why is LaMa better for large holes?  (Hint: Telea knows nothing about what a building wall "should" look like — it just propagates boundary values.)
- After inpainting, the atlas is fully filled.  Can you use the inpainted texture for any serious quantitative measurement (e.g. measuring building colour)?  Why or why not?

---

**Next → [Phase 8: Evaluation & Metrics](phase8_evaluation.ipynb)**